In [1]:
# ─── Imports ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

In [2]:
# ─── Load All 9 Tables ─────────────────────────────────────
path = '../data/raw/'

orders      = pd.read_csv(path + 'olist_orders_dataset.csv')
customers   = pd.read_csv(path + 'olist_customers_dataset.csv')
order_items = pd.read_csv(path + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(path + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(path + 'olist_products_dataset.csv')
sellers     = pd.read_csv(path + 'olist_sellers_dataset.csv')
category    = pd.read_csv(path + 'product_category_name_translation.csv')

print("All tables loaded!")
print(f"Orders: {orders.shape}")

All tables loaded!
Orders: (99441, 8)


In [3]:
len(orders)

99441

In [7]:
# ─── Define Target Variable ────────────────────────────────
# We keep ONLY orders with a final outcome:
# canceled = 1 (our positive class — what we want to predict)
# delivered = 0 (negative class — successful orders)
#
# We DROP: shipped, unavailable, invoiced, processing, created, approved
# WHY: These are in-progress orders with no final outcome yet.
# Training a model on incomplete outcomes = wrong predictions.

before = len(orders)
orders = orders[orders['order_status'].isin(['canceled', 'delivered'])]
after = len(orders)

orders['target'] = (orders['order_status'] == 'canceled').astype(int)

print(f"Orders before filter : {before:,}")
print(f"Orders after filter  : {after:,}")
print(f"Orders dropped       : {before - after:,}")
print(f"\nTarget distribution:")
print(orders['target'].value_counts())
print(f"\nCancellation rate: {orders['target'].mean()*100:.2f}%")

Orders before filter : 99,441
Orders after filter  : 97,103
Orders dropped       : 2,338

Target distribution:
target
0    96478
1      625
Name: count, dtype: int64

Cancellation rate: 0.64%


In [8]:
# ─── Convert Date Columns ──────────────────────────────────
# CSV format stores dates as strings. We convert to datetime
# so we can do date arithmetic for feature engineering later.

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns converted:")
print(orders[date_cols].dtypes)

Date columns converted:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [9]:
# ─── Aggregate Order Items ─────────────────────────────────
# One order can have MULTIPLE items (e.g. 3 products in one order)
# We cannot join directly — we'd get duplicate rows per order
# Solution: aggregate order_items to ONE row per order_id
#
# Features we extract:
# - total items in order
# - total freight value
# - number of unique sellers in one order
# - price stats (mean, max, sum)

items_agg = order_items.groupby('order_id').agg(
    item_count        = ('order_item_id', 'count'),
    total_price       = ('price', 'sum'),
    mean_price        = ('price', 'mean'),
    max_price         = ('price', 'max'),
    total_freight     = ('freight_value', 'sum'),
    unique_sellers    = ('seller_id', 'nunique')
).reset_index()

print(f"Items aggregated: {items_agg.shape}")
print(items_agg.head())

Items aggregated: (98666, 7)
                           order_id  item_count  total_price  mean_price  \
0  00010242fe8c5a6d1ba2dd792cb16214           1        58.90       58.90   
1  00018f77f2f0320c557190d7a144bdd3           1       239.90      239.90   
2  000229ec398224ef6ca0657da4fc703e           1       199.00      199.00   
3  00024acbcdf0a6daa1e931b038114c75           1        12.99       12.99   
4  00042b26cf59d7ce69dfabb4e55b4fd9           1       199.90      199.90   

   max_price  total_freight  unique_sellers  
0      58.90          13.29               1  
1     239.90          19.93               1  
2     199.00          17.87               1  
3      12.99          12.79               1  
4     199.90          18.14               1  


In [10]:
# ─── Aggregate Payments ────────────────────────────────────
# One order can have MULTIPLE payment methods
# (e.g. part credit card + part voucher)
# We aggregate to one row per order
#
# Features:
# - total payment value
# - number of payment installments
# - payment type (most common method used)
# - number of payment methods used (complexity indicator)

payments_agg = payments.groupby('order_id').agg(
    total_payment     = ('payment_value', 'sum'),
    max_installments  = ('payment_installments', 'max'),
    payment_methods   = ('payment_type', 'nunique'),
    payment_type      = ('payment_type', lambda x: x.mode()[0])
).reset_index()

print(f"Payments aggregated: {payments_agg.shape}")
print(payments_agg['payment_type'].value_counts())

Payments aggregated: (99440, 5)
payment_type
credit_card    76132
boleto         19784
voucher         1994
debit_card      1527
not_defined        3
Name: count, dtype: int64


In [11]:
# ─── Aggregate Reviews ─────────────────────────────────────
# Some orders have multiple reviews (rare but exists)
# We take the mean score and count of reviews per order
#
# NOTE: Cancelled orders often have NO review at all
# Missing review score will be a feature itself (NaN = no review)

reviews_agg = reviews.groupby('order_id').agg(
    review_score      = ('review_score', 'mean'),
    review_count      = ('review_id', 'count')
).reset_index()

print(f"Reviews aggregated: {reviews_agg.shape}")
print(f"Mean review score: {reviews_agg['review_score'].mean():.2f}")

Reviews aggregated: (98673, 3)
Mean review score: 4.09


In [17]:
products.head(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


In [18]:
# ─── Product Features via Order Items ──────────────────────
# We cannot join products directly to orders
# The path is: orders → order_items → products
# We join products to order_items first, then aggregate

# Add English category name
products = products.merge(category, on='product_category_name', how='left')

# Join products to order_items
items_with_products = order_items.merge(
    products[['product_id', 'product_weight_g', 
              'product_length_cm', 'product_height_cm',
              'product_width_cm', 'product_category_name_english']],
    on='product_id', how='left'
)

# Aggregate to order level
products_agg = items_with_products.groupby('order_id').agg(
    total_weight      = ('product_weight_g', 'sum'),
    mean_weight       = ('product_weight_g', 'mean'),
    top_category      = ('product_category_name_english', 
                         lambda x: x.mode()[0] if x.notna().any() else np.nan)
).reset_index()

print(f"Products aggregated: {products_agg.shape}")
print(products_agg.head())

Products aggregated: (98666, 4)
                           order_id  total_weight  mean_weight  \
0  00010242fe8c5a6d1ba2dd792cb16214        650.00       650.00   
1  00018f77f2f0320c557190d7a144bdd3      30000.00     30000.00   
2  000229ec398224ef6ca0657da4fc703e       3050.00      3050.00   
3  00024acbcdf0a6daa1e931b038114c75        200.00       200.00   
4  00042b26cf59d7ce69dfabb4e55b4fd9       3750.00      3750.00   

      top_category  
0       cool_stuff  
1         pet_shop  
2  furniture_decor  
3        perfumery  
4     garden_tools  


In [ ]:
# ─── Seller Features via Order Items ───────────────────────
# Seller state tells us seller location
# Multiple sellers in one order = complex fulfillment

items_with_sellers = order_items.merge(
    sellers[['seller_id', 'seller_state']],
    on='seller_id', how='left'
)

sellers_agg = items_with_sellers.groupby('order_id').agg(
    seller_count      = ('seller_id', 'nunique'),
    seller_state      = ('seller_state', lambda x: x.mode()[0])
).reset_index()

print(f"Sellers aggregated: {sellers_agg.shape}")
print(sellers_agg.head())

In [ ]:
# ─── Master Merge — All Tables into One ────────────────────
# We use LEFT JOIN on orders as the base table
# LEFT JOIN keeps all orders even if some have no items/payments/reviews
# This prevents rows from silently disappearing

master = orders.copy()

# Step 1: Add customer info
master = master.merge(
    customers[['customer_id', 'customer_state', 'customer_city']],
    on='customer_id', how='left'
)
print(f"After customers merge : {master.shape}")

# Step 2: Add aggregated items
master = master.merge(items_agg, on='order_id', how='left')
print(f"After items merge     : {master.shape}")

# Step 3: Add aggregated payments
master = master.merge(payments_agg, on='order_id', how='left')
print(f"After payments merge  : {master.shape}")

# Step 4: Add aggregated reviews
master = master.merge(reviews_agg, on='order_id', how='left')
print(f"After reviews merge   : {master.shape}")

# Step 5: Add product features
master = master.merge(products_agg, on='order_id', how='left')
print(f"After products merge  : {master.shape}")

# Step 6: Add seller features
master = master.merge(sellers_agg, on='order_id', how='left')
print(f"After sellers merge   : {master.shape}")

print(f"\nFinal master shape: {master.shape}")
print(f"Columns: {master.columns.tolist()}")

In [ ]:
# ─── Missing Values After Merge ────────────────────────────
# After merging, check what's missing and why
# Missing values from LEFT JOIN are expected for cancelled orders

missing = pd.DataFrame({
    'Missing Count': master.isnull().sum(),
    'Missing %': (master.isnull().sum() / len(master) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print("Missing values after master merge:")
print(missing)

In [ ]:
# ─── Handle Missing Values ─────────────────────────────────
# Different strategies for different columns:
#
# Numeric columns with missing: fill with 0 or median
# WHY 0 for counts: if an order has no items recorded = 0 items
# WHY median for continuous: avoids mean being pulled by outliers
#
# Categorical: fill with 'unknown'
# Review score: keep as NaN — missing review IS information
#   (cancelled orders have no review — this pattern is a feature)

# Numeric fill with 0
zero_fill_cols = ['item_count', 'total_price', 'mean_price', 'max_price',
                  'total_freight', 'unique_sellers', 'total_payment',
                  'max_installments', 'payment_methods', 'review_count',
                  'total_weight', 'mean_weight', 'seller_count']

for col in zero_fill_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0)

# Categorical fill with 'unknown'
cat_fill_cols = ['payment_type', 'customer_state', 'customer_city',
                 'seller_state', 'top_category']
for col in cat_fill_cols:
    if col in master.columns:
        master[col] = master[col].fillna('unknown')

# review_score: keep NaN but add a flag column
master['has_review'] = master['review_score'].notna().astype(int)
# Fill NaN review with median of available scores
master['review_score'] = master['review_score'].fillna(
    master['review_score'].median()
)

print("Missing values after treatment:")
remaining = master.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) == 0:
    print("No missing values remaining!")
else:
    print(remaining)

In [ ]:
# ─── Remove Duplicates ─────────────────────────────────────
before = len(master)
master = master.drop_duplicates(subset='order_id')
after = len(master)

print(f"Duplicates removed: {before - after:,}")
print(f"Final dataset shape: {master.shape}")
print(f"\nTarget distribution:")
print(master['target'].value_counts())
print(f"\nCancellation rate: {master['target'].mean()*100:.2f}%")

In [ ]:
# ─── Cleaning Summary ──────────────────────────────────────
print("=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)
print(f"Original orders table    : 99,441 rows")
print(f"After status filter      : {len(orders):,} rows")
print(f"Final master dataframe   : {len(master):,} rows")
print(f"Total columns            : {master.shape[1]}")
print(f"Target: canceled         : {master['target'].sum():,} ({master['target'].mean()*100:.2f}%)")
print(f"Target: delivered        : {(master['target']==0).sum():,} ({(master['target']==0).mean()*100:.2f}%)")
print("=" * 50)

In [ ]:
# ─── Save Master Dataset ───────────────────────────────────
import os
os.makedirs('../data/processed', exist_ok=True)

master.to_csv('../data/processed/master.csv', index=False)
print(f"Master dataset saved!")
print(f"Path: ../data/processed/master.csv")
print(f"Shape: {master.shape}")
master.head()

In [ ]:
# ─── Write src/cleaning.py ─────────────────────────────────
code = '''import pandas as pd
import numpy as np


def load_tables(path: str) -> dict:
    """Load all 9 Olist tables from raw data path."""
    return {
        "orders"      : pd.read_csv(path + "olist_orders_dataset.csv"),
        "customers"   : pd.read_csv(path + "olist_customers_dataset.csv"),
        "order_items" : pd.read_csv(path + "olist_order_items_dataset.csv"),
        "payments"    : pd.read_csv(path + "olist_order_payments_dataset.csv"),
        "reviews"     : pd.read_csv(path + "olist_order_reviews_dataset.csv"),
        "products"    : pd.read_csv(path + "olist_products_dataset.csv"),
        "sellers"     : pd.read_csv(path + "olist_sellers_dataset.csv"),
        "category"    : pd.read_csv(path + "product_category_name_translation.csv"),
    }


def build_master(tables: dict) -> pd.DataFrame:
    """Build master dataframe by joining all tables."""
    orders      = tables["orders"]
    customers   = tables["customers"]
    order_items = tables["order_items"]
    payments    = tables["payments"]
    reviews     = tables["reviews"]
    products    = tables["products"]
    sellers     = tables["sellers"]
    category    = tables["category"]

    # Define target
    orders = orders[orders["order_status"].isin(["canceled", "delivered"])]
    orders["target"] = (orders["order_status"] == "canceled").astype(int)

    # Convert dates
    date_cols = ["order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date"]
    for col in date_cols:
        orders[col] = pd.to_datetime(orders[col])

    # Aggregations
    items_agg = order_items.groupby("order_id").agg(
        item_count=("order_item_id","count"), total_price=("price","sum"),
        mean_price=("price","mean"), max_price=("price","max"),
        total_freight=("freight_value","sum"), unique_sellers=("seller_id","nunique")
    ).reset_index()

    payments_agg = payments.groupby("order_id").agg(
        total_payment=("payment_value","sum"),
        max_installments=("payment_installments","max"),
        payment_methods=("payment_type","nunique"),
        payment_type=("payment_type", lambda x: x.mode()[0])
    ).reset_index()

    reviews_agg = reviews.groupby("order_id").agg(
        review_score=("review_score","mean"),
        review_count=("review_id","count")
    ).reset_index()

    products = products.merge(category, on="product_category_name", how="left")
    items_with_products = order_items.merge(
        products[["product_id","product_weight_g","product_category_name_english"]],
        on="product_id", how="left"
    )
    products_agg = items_with_products.groupby("order_id").agg(
        total_weight=("product_weight_g","sum"),
        mean_weight=("product_weight_g","mean"),
        top_category=("product_category_name_english",
                      lambda x: x.mode()[0] if x.notna().any() else np.nan)
    ).reset_index()

    items_with_sellers = order_items.merge(
        sellers[["seller_id","seller_state"]], on="seller_id", how="left"
    )
    sellers_agg = items_with_sellers.groupby("order_id").agg(
        seller_count=("seller_id","nunique"),
        seller_state=("seller_state", lambda x: x.mode()[0])
    ).reset_index()

    # Master merge
    master = orders.copy()
    master = master.merge(customers[["customer_id","customer_state","customer_city"]], on="customer_id", how="left")
    master = master.merge(items_agg, on="order_id", how="left")
    master = master.merge(payments_agg, on="order_id", how="left")
    master = master.merge(reviews_agg, on="order_id", how="left")
    master = master.merge(products_agg, on="order_id", how="left")
    master = master.merge(sellers_agg, on="order_id", how="left")

    # Handle missing values
    zero_cols = ["item_count","total_price","mean_price","max_price",
                 "total_freight","unique_sellers","total_payment",
                 "max_installments","payment_methods","review_count",
                 "total_weight","mean_weight","seller_count"]
    for col in zero_cols:
        if col in master.columns:
            master[col] = master[col].fillna(0)

    cat_cols = ["payment_type","customer_state","customer_city","seller_state","top_category"]
    for col in cat_cols:
        if col in master.columns:
            master[col] = master[col].fillna("unknown")

    master["has_review"] = master["review_score"].notna().astype(int)
    master["review_score"] = master["review_score"].fillna(master["review_score"].median())
    master = master.drop_duplicates(subset="order_id")

    return master
'''

with open('../src/cleaning.py', 'w') as f:
    f.write(code)
print("src/cleaning.py created!")

In [ ]:
#Virat kohli -18
#Rajat patidar - 18
#Salt -14
#Hazelwod -14
#Krunal -4 
#Bhuvi - 4
